In [ ]:
# 1. GPU check
!nvidia-smi

# 2. Install dependencies
!pip -q install -U transformers datasets sentencepiece accelerate huggingface_hub python-dotenv feedparser beautifulsoup4 sacrebleu rouge-score gradio



# 3. Clone or pull repo
from getpass import getpass
token = getpass("Enter your GitHub PAT: ")
USERNAME = "sukritichawla"
REPO = "legaldocs_ST"
REPO_URL = f"https://{token}@github.com/{USERNAME}/{REPO}.git"
!git clone $REPO_URL


# 4. Go inside repo
%cd /content/legaldocs_STw

# 5. Hugging Face login
from huggingface_hub import login
login()

Sat Sep 27 09:40:20 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install transformers datasets sacrebleu sentencepiece torch --quiet


In [ ]:
from datasets import load_dataset

# Load OPUS100 dataset (English-Malayalam parallel data)
dataset = load_dataset("opus100", "en-ml")

# Reduce size → take only 10k training and 1k test samples
small_train = dataset["train"].shuffle(seed=42).select(range(160000))
small_test  = dataset["test"].shuffle(seed=42).select(range(2000))

print(small_train[0])


{'translation': {'en': "I've got a car, we can drive around, we'll have a lot of fun!", 'ml': 'പിന്നെ, എന്\u200dറെ കാറിൽ നമുക്ക് കറങ്ങാൻ പോകാം, അങ്ങിനെ ഒരു പാട് രസമായിരിക്കും!'}}


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "Helsinki-NLP/opus-mt-en-ml"   # already trained EN→ML model

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/449k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/614k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/229M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [ ]:
def preprocess_function(examples):
    # Extract English and Malayalam from the translation field
    inputs = [ex["en"] for ex in examples["translation"]]
    targets = [ex["ml"] for ex in examples["translation"]]

    model_inputs = tokenizer(inputs, max_length=64, truncation=True)

    # Tokenize targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=64, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_data = small_train.map(preprocess_function, batched=True)
test_data  = small_test.map(preprocess_function, batched=True)


Map:   0%|          | 0/160000 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4007: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [ ]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

args = Seq2SeqTrainingArguments(
    output_dir="en-ml-small",
    eval_strategy="steps",
    eval_steps=500,
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=3,
    predict_with_generate=True,
    logging_dir="./logs",
    logging_steps=100,
    report_to="none",
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)


In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

# Build trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_data,
    eval_dataset=test_data,
    processing_class=tokenizer,
    data_collator=data_collator,
)

# Train
trainer.train()


Step,Training Loss,Validation Loss
500,0.841000,2.339926
1000,0.876400,2.364956
1500,1.453700,2.314139
2000,1.487500,2.312643


Step,Training Loss,Validation Loss
500,0.841000,2.339926
1000,0.876400,2.364956
1500,1.453700,2.314139
2000,1.487500,2.312643
2500,1.411600,2.298393
3000,1.523000,2.292558
3500,1.447200,2.283708
4000,1.471100,2.280602
4500,1.426000,2.281869
5000,1.431600,2.271689


TrainOutput(global_step=60000, training_loss=1.19337306543986, metrics={'train_runtime': 6866.4691, 'train_samples_per_second': 69.905, 'train_steps_per_second': 8.738, 'total_flos': 4719781637259264.0, 'train_loss': 1.19337306543986, 'epoch': 3.0})

In [ ]:
# Save model and tokenizer
model.save_pretrained("en-ml-finetuned")
tokenizer.save_pretrained("en-ml-finetuned")

print("Model saved locally in folder 'en-ml-finetuned'")



Model saved locally in folder 'en-ml-finetuned'


In [ ]:
import numpy as np
import sacrebleu

def compute_bleu(trainer, dataset):
    preds = trainer.predict(dataset)
    decoded_preds = tokenizer.batch_decode(preds.predictions, skip_special_tokens=True)
    labels = np.where(preds.label_ids != -100, preds.label_ids, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    bleu = sacrebleu.corpus_bleu(decoded_preds, [decoded_labels])
    return bleu.score

bleu_score = compute_bleu(trainer, test_data)
print("BLEU score on test set:", bleu_score)


BLEU score on test set: 8.970686068715429


In [ ]:
import torch

text = "Do you remember this"

# Get device (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move model to device
model.to(device)

# Tokenize and move inputs to device
inputs = tokenizer(text, return_tensors="pt").to(device)

# Generate translation
outputs = model.generate(**inputs, max_length=40)

print("English:", text)
print("Malayalam:", tokenizer.decode(outputs[0], skip_special_tokens=True))


English: Do you remember this
Malayalam: നിനക്കിത് ഓര്മ്മയുണ്ടോ?
